# 01 รับข้อมูลและทำความเข้าใจ
ก่อนเริ่ม: รับไฟล์จากอาจารย์และเก็บต้นฉบับแยกไว้ ถ้ายังไม่มีไฟล์ให้อ่านคำถามและเตรียมพจนานุกรม ไม่ต้องหาชุดอื่นมาแทน

สิ่งที่จะได้จากงานของสมาชิก: ความหมายหนึ่งแถว target หน่วย บริบทการทำนาย แหล่งข้อมูลและสิทธิ์ที่ยืนยันแล้ว


In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents,
              Path('/content/Bicycle_Rental_Prediction')]
ROOT = next((p for p in candidates if (p / 'config/project.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('เปิด notebook จาก repository หรือทำขั้นตอนเตรียม Colab ใน START_HERE_TH ก่อน')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from starter_support import load_config, configuration_problems, resolve_dataset_path
config = load_config(ROOT)
problems = configuration_problems(config)
if problems:
    raise ValueError('\n'.join(problems))
print('อ่าน config ได้แล้ว ค่า null เป็นสถานะรอยืนยัน ไม่ใช่ข้อมูลที่ต้องเดา')


## ตั้งเฉพาะเส้นทางก่อน
แก้ dataset.path ใน config/project.json ให้ชี้ CSV ที่ได้รับ ใช้เส้นทางภายใน data/ ด้วย / เช่น data/raw/ชื่อไฟล์จริง.csv ยืนยัน encoding/delimiter หากไฟล์ไม่ใช่ CSV ให้ตกลงตัวอ่านหรือวิธีแปลงและเก็บต้นฉบับไว้


In [ ]:
df = None
path = resolve_dataset_path(config, ROOT)
if path is None:
    print('ยังไม่ได้ตั้ง dataset.path: อ่านโจทย์และคำถามก่อนได้ ยังไม่มีข้อมูลให้วิเคราะห์')
elif not path.is_file():
    print('ยังไม่พบไฟล์: วาง dataset จากอาจารย์ใน data/raw ตาม config')
else:
    import pandas as pd
    df = pd.read_csv(path, encoding=config['dataset']['encoding'],
                     sep=config['dataset']['delimiter'])
    print('อ่านไฟล์ได้แล้ว ต้องตรวจความหมายและคุณภาพก่อนใช้งาน')


In [ ]:
if df is not None:
    print('จำนวนแถวและคอลัมน์:', df.shape)
    print('รายชื่อคอลัมน์:', list(df.columns))
    print(df.dtypes)
else:
    print('ยังไม่มีข้อมูลจริงให้สรุป schema')


## สมาชิกลงมือทำ
1. อ่าน data dictionary จากอาจารย์ถ้ามี แล้วเติม evidence/data_dictionary.csv
2. ยืนยันว่าแถวหนึ่งเป็นชั่วโมง วัน หรือสถานี และช่วงเวลาใด
3. ระบุ target/หน่วย และอธิบายว่า input แต่ละตัวรู้เมื่อใด
4. แยก identifiers ส่วนประกอบของ target และข้อมูลหลังเหตุการณ์ออกจาก feature
5. บันทึกแหล่งข้อมูล สิทธิ์เผยแพร่ และคำถามที่ยังไม่รู้ใน decisions.md
6. เติม prediction ใน config หลังยืนยันความหมาย ห้ามจัด numeric/category จากตัวเลขในไฟล์อย่างเดียว

เขียนคำตอบและโค้ดสำรวจเพิ่มเติมด้วยตนเองด้านล่าง


In [ ]:
# เพิ่มการสำรวจที่จำเป็นหลังได้รับไฟล์ โดยเก็บข้อมูลอ่อนไหวไว้ในเครื่อง
# เขียนเหตุผลเลือก target และ feature ใน markdown cell ของตนเอง


## จุดตรวจโดยเพื่อน
อธิบายคำถามที่จะทำนายได้ในหนึ่งประโยค ระบุหน่วยและเวลาที่ input พร้อมใช้ ไม่มี target ปนใน X แล้วรัน scripts/check_setup.py --require-data

ตัวตรวจชื่อคอลัมน์ไม่สามารถตัดสินได้ว่า feature รั่วหรือหน่วยถูก ต้องตรวจด้วยคน


## ก่อนส่งต่อ
เก็บข้อสังเกตและหลักฐานที่เผยแพร่ได้ไว้ใน evidence/results ให้เพื่อนอีกคนตรวจและลองทำซ้ำ จากนั้นล้าง outputs และ execution counts ก่อน commit notebook อย่าใส่ผลสมมติเป็นผลทดลองจริง
